# 17 · EU LNG Send-Out & Terminal Utilization Analysis

**Data source:** GIE ALSI+ API (https://alsi.gie.eu) — LNG terminal daily data for Belgium, Spain, France, Greece, Italy, Lithuania, Netherlands, Poland, Portugal.

This notebook applies the analytical framework used by **Bruegel**, **IEEFA**, and **ACER** in their LNG market monitors. The central metric is the **utilization rate**: actual daily send-out as a share of technical send-out capacity. This framing separates *physical* capacity from *economic* throughput — a terminal capable of 500 GWh/day may operate at 200 GWh/day (40%) when the LNG–TTF spread is too narrow to attract cargoes.

**Key benchmarks from industry research:**
- **EU aggregate utilization ~42–55%** (2024–2026) — ACER Market Monitoring Report, Bruegel LNG tracker
- **Strong terminal dispersion**: individual terminals range 20–90%, driven by contract structure, source flexibility, and proximity to demand centres
- **Economic driver**: higher TTF prices and wider LNG–TTF spreads pull more cargoes into Europe → higher utilization
- **Seasonal pattern**: utilization often rises in injection season (Apr–Sep) as storage refill demand absorbs LNG supply

## Cell 0 — Setup
Load data, initialise ALSIClient, fetch per-country data if needed.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"✅ Root: {_c}"); break

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — paste your ALSI API key below
# ═══════════════════════════════════════════════════════════════
API_KEY    = "paste_key_here"
START_DATE = "2019-01-01"
# ═══════════════════════════════════════════════════════════════

from src.agsi_client.alsi_client import ALSIClient, EU_COUNTRIES

client = ALSIClient(api_key=API_KEY, ssl_verify=False)

def _strip_tz(df):
    """Remove timezone from DatetimeIndex, ensure DatetimeIndex type."""
    if not df.empty and hasattr(df.index, "tz") and df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df.index = pd.to_datetime(df.index)
    return df

# ── EU aggregate LNG ───────────────────────────────────────────
lng_path = Path("data/processed/eu_lng_full.parquet")
if lng_path.exists():
    lng_df = pd.read_parquet(lng_path)
    print(f"✅ Loaded EU LNG aggregate: {len(lng_df)} rows from {lng_path}")
else:
    print("Fetching EU LNG aggregate from ALSI API…")
    lng_df = client.get_eu_aggregate(start=START_DATE)
    if not lng_df.empty:
        lng_path.parent.mkdir(parents=True, exist_ok=True)
        lng_df.to_parquet(lng_path)
        print(f"✅ Saved {len(lng_df)} rows → {lng_path}")
lng_df = _strip_tz(lng_df)

# Parse any send-out capacity fields that the API may return as strings
for col in ["dtrs", "dtrsCapacity", "sendOutCapacity"]:
    if col in lng_df.columns:
        lng_df[col] = pd.to_numeric(lng_df[col], errors="coerce")

# ── Per-country LNG ─────────────────────────────────────────────
country_path = Path("data/processed/lng_by_country.parquet")
if country_path.exists():
    lng_country = pd.read_parquet(country_path)
    print(f"✅ Loaded per-country LNG data: {len(lng_country)} rows")
else:
    print("Fetching per-country LNG data from ALSI API…")
    frames = []
    for c in EU_COUNTRIES:
        df_c = client.get_country(c, start=START_DATE)
        if not df_c.empty:
            frames.append(df_c)
            print(f"  {c}: {len(df_c)} rows")
    if frames:
        lng_country = pd.concat(frames).sort_index()
        country_path.parent.mkdir(parents=True, exist_ok=True)
        lng_country.to_parquet(country_path)
        print(f"✅ Saved per-country data → {country_path}")
    else:
        lng_country = pd.DataFrame()
        print("⚠️ No per-country data fetched")
if not lng_country.empty:
    lng_country = _strip_tz(lng_country)
    for col in ["dtrs", "dtrsCapacity", "sendOutCapacity"]:
        if col in lng_country.columns:
            lng_country[col] = pd.to_numeric(lng_country[col], errors="coerce")

# ── TTF forward curve ──────────────────────────────────────────
ttf_path = Path("data/raw/ttf_curve.csv")
if ttf_path.exists():
    ttf_df = pd.read_csv(ttf_path, index_col=0, parse_dates=True)
    ttf_df = _strip_tz(ttf_df)
    print(f"✅ TTF curve loaded: {len(ttf_df)} rows, columns: {list(ttf_df.columns[:5])}…")
else:
    ttf_df = pd.DataFrame()
    print("⚠️ TTF curve not found at data/raw/ttf_curve.csv")

print(f"\nLNG EU columns : {list(lng_df.columns)}")
print(f"Date range     : {lng_df.index.min().date() if not lng_df.empty else 'n/a'} → "
      f"{lng_df.index.max().date() if not lng_df.empty else 'n/a'}")
if not lng_country.empty:
    print(f"Countries      : {sorted(lng_country['country'].unique())}")

## Cell 1 — EU Aggregate Send-Out Time Series

Daily EU LNG send-out (GWh/day) with 30-day rolling average and year-on-year comparison overlays for the two prior calendar years. This gives the clearest view of structural trends and seasonal patterns in LNG throughput.

In [ ]:
if lng_df.empty or "sendOut" not in lng_df.columns:
    print("⚠️ No send-out data available — check API key and network.")
else:
    so = lng_df["sendOut"].dropna()
    so_7d  = so.rolling(7,  min_periods=3).mean()
    so_30d = so.rolling(30, min_periods=7).mean()

    today  = so.index[-1]
    yr_cur = today.year
    yr1    = so[so.index.year == yr_cur - 1]
    yr2    = so[so.index.year == yr_cur - 2]

    def _align_year(s, target_year):
        """Shift a year-slice's dates to target_year for overlay."""
        s = s.copy()
        s.index = s.index.map(lambda d: d.replace(year=target_year))
        return s

    fig = go.Figure()

    # Light daily
    fig.add_trace(go.Scatter(
        x=so.index, y=so, name="Daily", mode="lines",
        line=dict(color="#bfdbfe", width=0.7), opacity=0.55, showlegend=True
    ))
    # 7d rolling
    fig.add_trace(go.Scatter(
        x=so_7d.index, y=so_7d, name="7d avg",
        line=dict(color="#93c5fd", width=1.3)
    ))
    # 30d rolling — primary trend line
    fig.add_trace(go.Scatter(
        x=so_30d.index, y=so_30d, name="30d avg",
        line=dict(color="#2563eb", width=2.4)
    ))

    # YoY overlays aligned to current year
    if not yr1.empty:
        fig.add_trace(go.Scatter(
            x=_align_year(yr1, yr_cur).index,
            y=_align_year(yr1, yr_cur).values,
            name=str(yr_cur - 1),
            line=dict(color="#f97316", width=1.6, dash="dot")
        ))
    if not yr2.empty:
        fig.add_trace(go.Scatter(
            x=_align_year(yr2, yr_cur).index,
            y=_align_year(yr2, yr_cur).values,
            name=str(yr_cur - 2),
            line=dict(color="#94a3b8", width=1.2, dash="dot")
        ))

    fig.update_layout(
        template="plotly_white", height=440,
        title="EU LNG Send-Out Rate (GWh/day) — Daily + 30d Rolling + YoY",
        yaxis_title="GWh/day", xaxis_title="",
        legend=dict(orientation="h", y=-0.15)
    )
    fig.show()

    latest  = so.iloc[-1]
    prev_yr = yr1.iloc[-1] if not yr1.empty else np.nan
    print(f"Latest send-out  : {latest:,.0f} GWh/day  ({so.index[-1].date()})")
    print(f"Same date -1yr   : {prev_yr:,.0f} GWh/day")
    print(f"YoY change       : {(latest/prev_yr - 1)*100:+.1f}%")
    print(f"30d rolling avg  : {so_30d.dropna().iloc[-1]:,.0f} GWh/day")
    print(f"Historical max   : {so.max():,.0f} GWh/day  ({so.idxmax().date()})")

## Cell 2 — Utilization Rate

**Utilization rate = sendOut / send-out capacity × 100**

This is the metric industry trackers (Bruegel, IEEFA, ACER) use as the primary indicator of LNG market tightness. Raw volumes tell only half the story; utilization rate reveals how much of Europe's regasification infrastructure is actually being used.

**Capacity note:** The ALSI API returns `dtrs` (Daily Technical Recommended Sendout, GWh/day) when available. If the field is absent in the response, the notebook falls back to a 365-day rolling 95th-percentile proxy — a common approximation when design-capacity data is missing. The proxy will tend to underestimate true capacity, causing utilization estimates to be slightly overstated; interpret with care and the label is shown on the chart.

**Reference lines** from industry research:
- ~42% — EU aggregate annual average utilization, 2024 (ACER Market Monitoring Report)
- ~55% — Higher-utilization periods including injection-season peaks

In [ ]:
if lng_df.empty or "sendOut" not in lng_df.columns:
    print("⚠️ No send-out data available.")
else:
    so = lng_df["sendOut"].dropna()

    # ── Determine capacity denominator ──────────────────────────
    DTRS_COLS = ["dtrs", "dtrsCapacity", "sendOutCapacity"]
    cap_col_found = next((c for c in DTRS_COLS if c in lng_df.columns
                          and lng_df[c].notna().sum() > 50), None)

    if cap_col_found:
        cap       = lng_df[cap_col_found].reindex(so.index)
        cap_label = f"{cap_col_found} (API — design capacity)"
        print(f"✅ Using '{cap_col_found}' from API as send-out capacity denominator")
    else:
        # Rolling 95th percentile proxy
        cap       = so.rolling(365, min_periods=90).quantile(0.95)
        cap_label = "rolling 365d 95th-pct (proxy — true capacity likely higher)"
        print("ℹ️  No dtrs column found — using 365-day rolling 95th percentile as capacity proxy")
        print("    Note: this may overstate utilization vs. true design capacity")

    util    = (so / cap * 100).clip(upper=105)
    util_30d = util.rolling(30, min_periods=7).mean()

    BENCH_LOW  = 42
    BENCH_HIGH = 55

    today    = util.index[-1]
    yr_cur   = today.year
    yr1_util = util[util.index.year == yr_cur - 1]

    def _align_year(s, target_year):
        s = s.copy()
        s.index = s.index.map(lambda d: d.replace(year=target_year))
        return s

    fig = go.Figure()

    # Daily utilization (faint background)
    fig.add_trace(go.Scatter(
        x=util.index, y=util, name="Daily util.",
        line=dict(color="#c7d2fe", width=0.7), opacity=0.4
    ))
    # 30d rolling — filled
    fig.add_trace(go.Scatter(
        x=util_30d.index, y=util_30d, name="30d avg",
        line=dict(color="#6366f1", width=2.2),
        fill="tozeroy", fillcolor="rgba(99,102,241,0.07)"
    ))
    # YoY -1yr overlay
    if not yr1_util.empty:
        fig.add_trace(go.Scatter(
            x=_align_year(yr1_util, yr_cur).index,
            y=_align_year(yr1_util, yr_cur).values,
            name=str(yr_cur - 1),
            line=dict(color="#f97316", width=1.5, dash="dot")
        ))

    # Benchmark reference lines
    fig.add_hline(y=BENCH_LOW, line_dash="dash", line_color="#ef4444", line_width=1.2,
                  annotation_text=f"~{BENCH_LOW}% — EU avg 2024 (ACER)",
                  annotation_position="bottom right", annotation_font_size=10)
    fig.add_hline(y=BENCH_HIGH, line_dash="dash", line_color="#f97316", line_width=1.2,
                  annotation_text=f"~{BENCH_HIGH}% — injection-season peaks",
                  annotation_position="bottom right", annotation_font_size=10)

    fig.update_layout(
        template="plotly_white", height=460,
        title=f"EU LNG Utilization Rate (%) — capacity basis: {cap_label}",
        yaxis_title="Utilization (%)", xaxis_title="",
        yaxis=dict(range=[0, min(util.quantile(0.99) * 1.1, 110)]),
        legend=dict(orientation="h", y=-0.15)
    )
    fig.show()

    cur_util  = util.iloc[-1]
    prev_util = yr1_util.iloc[-1] if not yr1_util.empty else np.nan
    ytd       = util[util.index.year == yr_cur]

    print(f"Current utilization : {cur_util:.1f}%  ({util.index[-1].date()})")
    print(f"Same date -1yr      : {prev_util:.1f}%")
    print(f"YoY change          : {cur_util - prev_util:+.1f} ppt")
    print(f"30d avg             : {util_30d.dropna().iloc[-1]:.1f}%")
    print(f"YTD {yr_cur} average : {ytd.mean():.1f}%")
    print(f"\nCapacity basis      : {cap_label}")

## Cell 3 — Per-Country / Per-Terminal Breakdown

Industry reports consistently emphasise **terminal dispersion** as a key feature of the EU LNG market: some terminals (typically Spanish and French facilities with long-term supply contracts) operate above 70% utilization, while others (often smaller, merchant-mode terminals) run below 30%. This asymmetry matters because EU-aggregate figures can mask significant structural differences.

The chart below mirrors the approach taken in IEEFA's European LNG terminal utilization tracker: rank countries by current utilization, distinguish high/mid/low tiers.

In [ ]:
if lng_country.empty or "sendOut" not in lng_country.columns:
    print("⚠️ Per-country data not available — re-run Cell 0 with a valid API key.")
else:
    # ── Get most-recent period (last 30 days with data) for each country ──
    recent_cutoff = lng_country.index.max() - pd.Timedelta(days=30)
    recent        = lng_country[lng_country.index >= recent_cutoff]

    # Determine per-country send-out capacity
    DTRS_COLS = ["dtrs", "dtrsCapacity", "sendOutCapacity"]
    cap_col_c = next((c for c in DTRS_COLS if c in lng_country.columns
                      and lng_country[c].notna().sum() > 50), None)

    if cap_col_c:
        cap_src = "dtrs (API)"
    else:
        # Per-country 365-day rolling 95th pct proxy
        cap_src = "rolling 95th-pct proxy"
        for c_code in lng_country["country"].unique():
            mask = lng_country["country"] == c_code
            lng_country.loc[mask, "_cap_proxy"] = (
                lng_country.loc[mask, "sendOut"]
                .rolling(365, min_periods=60).quantile(0.95)
            )
        cap_col_c = "_cap_proxy"

    # Per-country recent stats
    country_stats = []
    for c_code, grp in recent.groupby("country"):
        so_mean  = grp["sendOut"].mean()
        cap_mean = grp[cap_col_c].mean() if cap_col_c in grp.columns else np.nan
        util_cur = so_mean / cap_mean * 100 if cap_mean > 0 else np.nan

        # 1-year average utilization
        all_c    = lng_country[lng_country["country"] == c_code]
        yr_cutoff= lng_country.index.max() - pd.Timedelta(days=365)
        yr_data  = all_c[all_c.index >= yr_cutoff]
        yr_cap   = yr_data[cap_col_c].mean() if cap_col_c in yr_data.columns else np.nan
        util_1yr = (yr_data["sendOut"].mean() / yr_cap * 100) if yr_cap > 0 else np.nan

        country_stats.append({
            "Country": c_code,
            "Send-out GWh/d": round(so_mean, 0),
            "Capacity GWh/d": round(cap_mean, 0),
            "Util % (current)": round(util_cur, 1),
            "Util % (1yr avg)": round(util_1yr, 1),
        })

    stats_df = (pd.DataFrame(country_stats)
                .dropna(subset=["Util % (current)"])
                .sort_values("Util % (current)", ascending=False)
                .reset_index(drop=True))

    # ── Bar chart ────────────────────────────────────────────────
    def _util_color(u):
        if u >= 70:   return "#22c55e"    # high
        elif u >= 30: return "#f97316"    # mid
        else:         return "#ef4444"    # low

    colors = [_util_color(u) for u in stats_df["Util % (current)"]]

    fig = go.Figure(go.Bar(
        x=stats_df["Country"],
        y=stats_df["Util % (current)"],
        marker_color=colors,
        text=[f"{v:.0f}%" for v in stats_df["Util % (current)"]],
        textposition="outside",
        name="Current util. %"
    ))
    fig.add_trace(go.Scatter(
        x=stats_df["Country"],
        y=stats_df["Util % (1yr avg)"],
        name="1-yr avg util. %",
        mode="markers",
        marker=dict(symbol="diamond", size=10, color="#6366f1",
                    line=dict(color="#4338ca", width=1.5))
    ))
    fig.add_hline(y=42, line_dash="dash", line_color="#94a3b8", line_width=1,
                  annotation_text="EU avg 42%", annotation_position="bottom right",
                  annotation_font_size=10)

    # Tier annotations
    fig.add_hrect(y0=70, y1=105, fillcolor="rgba(34,197,94,0.06)", line_width=0,
                  annotation_text="High (>70%)", annotation_position="top right",
                  annotation_font_size=10)
    fig.add_hrect(y0=0, y1=30, fillcolor="rgba(239,68,68,0.06)", line_width=0,
                  annotation_text="Low (<30%)", annotation_position="bottom right",
                  annotation_font_size=10)

    fig.update_layout(
        template="plotly_white", height=440,
        title=f"EU LNG Utilization Rate by Country — current 30d avg vs 1-yr avg [{cap_src}]",
        yaxis_title="Utilization (%)", xaxis_title="Country",
        yaxis=dict(range=[0, min(stats_df["Util % (current)"].max() * 1.18, 110)]),
        legend=dict(orientation="h", y=-0.14)
    )
    fig.show()

    # ── Summary table ─────────────────────────────────────────────
    print("\n" + "─"*68)
    print(f"  {'Country':<10} {'Send-out':>12} {'Capacity':>12} {'Util%':>8} {'1yr avg%':>10}")
    print(f"  {'':─<10} {'GWh/d':>12} {'GWh/d':>12} {'':>8} {'':>10}")
    for _, row in stats_df.iterrows():
        tier = ("▲ high" if row["Util % (current)"] >= 70
                else ("▼ low" if row["Util % (current)"] < 30 else ""))
        print(f"  {row['Country']:<10} {row['Send-out GWh/d']:>12,.0f} "
              f"{row['Capacity GWh/d']:>12,.0f} "
              f"{row['Util % (current)']:>7.1f}% "
              f"{row['Util % (1yr avg)']:>9.1f}%  {tier}")
    print("─"*68)
    high_cnt = (stats_df["Util % (current)"] >= 70).sum()
    low_cnt  = (stats_df["Util % (current)"] <  30).sum()
    print(f"  High-utilization (≥70%): {high_cnt}  |  Low-utilization (<30%): {low_cnt}")
    print(f"  Capacity basis: {cap_src}")

## Cell 4 — Seasonal Utilization Pattern

Bruegel's LNG tracker and ACER's quarterly reports consistently show a seasonal pattern: EU LNG utilization tends to be higher in the **injection season** (April–September) than in the withdrawal season (October–March), contrary to the intuition that higher winter gas demand would pull in more LNG. The explanation is that summer storage-filling creates a sustained demand for gas supply that LNG can profitably serve — while in winter, pipeline gas from Russia (historically) or Norway fills most of the demand, sometimes leaving LNG terminals underutilized despite high spot prices.

The box plot below shows the **distribution** of daily utilization by calendar month across all available years, making the seasonal pattern explicit.

In [ ]:
if lng_df.empty or "sendOut" not in lng_df.columns:
    print("⚠️ No send-out data available.")
else:
    so = lng_df["sendOut"].dropna()

    # Rebuild utilization from EU aggregate
    DTRS_COLS = ["dtrs", "dtrsCapacity", "sendOutCapacity"]
    cap_col_found = next((c for c in DTRS_COLS if c in lng_df.columns
                          and lng_df[c].notna().sum() > 50), None)
    if cap_col_found:
        cap = lng_df[cap_col_found].reindex(so.index)
    else:
        cap = so.rolling(365, min_periods=90).quantile(0.95)

    util = (so / cap * 100).clip(upper=105).dropna()
    util = util[util.index >= so.index.min() + pd.Timedelta(days=365)]  # drop first year (rolling warmup)

    month_names = ["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"]

    fig = go.Figure()
    INJ_COLOR = "#22c55e"   # green for injection season
    WDR_COLOR = "#6366f1"   # indigo for withdrawal season

    for m in range(1, 13):
        vals = util[util.index.month == m].values
        color = INJ_COLOR if 4 <= m <= 9 else WDR_COLOR
        fig.add_trace(go.Box(
            y=vals, name=month_names[m-1],
            marker_color=color,
            line_color=color,
            boxmean="sd",
            showlegend=(m in [4, 10]),  # one entry per season for legend
        ))

    # Custom legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                             marker=dict(size=10, color=INJ_COLOR, symbol="square"),
                             name="Injection season (Apr–Sep)"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                             marker=dict(size=10, color=WDR_COLOR, symbol="square"),
                             name="Withdrawal season (Oct–Mar)"))

    fig.update_layout(
        template="plotly_white", height=460,
        title="EU LNG Utilization Rate by Calendar Month — Distribution Across All Years",
        yaxis_title="Utilization (%)", xaxis_title="Month",
        showlegend=True,
        legend=dict(orientation="h", y=-0.14),
        boxmode="overlay"
    )
    fig.show()

    # Season averages
    inj_months = util[util.index.month.isin(range(4, 10))]
    wdr_months = util[~util.index.month.isin(range(4, 10))]

    print("Seasonal utilization summary:")
    print(f"  Injection season (Apr–Sep) : {inj_months.mean():.1f}%  median {inj_months.median():.1f}%")
    print(f"  Withdrawal season (Oct–Mar): {wdr_months.mean():.1f}%  median {wdr_months.median():.1f}%")
    diff = inj_months.mean() - wdr_months.mean()
    direction = "higher" if diff > 0 else "lower"
    print(f"  Injection-season utilization is {abs(diff):.1f} ppt {direction} than withdrawal season")

    # Monthly medians
    print("\nMonthly median utilization (%)")
    print("  " + "  ".join(f"{month_names[m-1]:>5}" for m in range(1,13)))
    medians = [f"{util[util.index.month == m].median():>5.1f}" for m in range(1,13)]
    print("  " + "  ".join(medians))

## Cell 5 — LNG–TTF Economic Driver

The central hypothesis in LNG market analysis is that **higher TTF prices attract more LNG to Europe** — when TTF is elevated relative to alternative LNG destinations (JKM for Asia, HH for the US), European buyers outbid competitors for flexible LNG cargoes, and utilization rises.

Without JKM (Asian spot) data in this dataset, we use TTF M1 price as a proxy for the European demand signal. A positive correlation between TTF and utilization is consistent with the economic-pull hypothesis. The rolling correlation shows how this relationship varies through time — it tends to strengthen during high-volatility regimes (2021–2022) and weaken when LNG supply is structurally committed or constrained.

**Data limitation:** A proper LNG–TTF spread requires JKM or FOB LNG price data (e.g. Platts JKM, DES NWE). If available in future notebooks, replace the TTF-only scatter with a TTF–JKM spread chart for a cleaner economic signal.

In [ ]:
if lng_df.empty or ttf_df.empty or "sendOut" not in lng_df.columns:
    print("⚠️ Need both LNG and TTF data. Check Cell 0.")
else:
    so = lng_df["sendOut"].dropna()
    DTRS_COLS = ["dtrs", "dtrsCapacity", "sendOutCapacity"]
    cap_col_found = next((c for c in DTRS_COLS if c in lng_df.columns
                          and lng_df[c].notna().sum() > 50), None)
    if cap_col_found:
        cap = lng_df[cap_col_found].reindex(so.index)
    else:
        cap = so.rolling(365, min_periods=90).quantile(0.95)
    util = (so / cap * 100).clip(upper=105)

    # Align TTF M1 with utilization
    ttf_col = "M1" if "M1" in ttf_df.columns else ttf_df.columns[0]
    merged = (util.rename("util")
              .to_frame()
              .join(ttf_df[[ttf_col]].rename(columns={ttf_col: "ttf_m1"}), how="inner")
              .dropna()
              .sort_index())

    if merged.empty:
        print("⚠️ No overlapping dates between LNG utilization and TTF data.")
    else:
        # ── Scatter ────────────────────────────────────────────────
        corr_total = merged["util"].corr(merged["ttf_m1"])

        fig_scatter = go.Figure(go.Scatter(
            x=merged["ttf_m1"], y=merged["util"],
            mode="markers",
            marker=dict(
                color=merged.index.year,
                colorscale="Blues",
                size=3.5, opacity=0.55,
                showscale=True,
                colorbar=dict(title="Year", thickness=12)
            ),
            text=merged.index.strftime("%Y-%m-%d"),
            hovertemplate="TTF: %{x:.1f} €/MWh<br>Util: %{y:.1f}%<br>%{text}<extra></extra>"
        ))

        # Trend line (OLS)
        from numpy.polynomial.polynomial import polyfit as np_polyfit
        try:
            x_vals = merged["ttf_m1"].values
            y_vals = merged["util"].values
            c, m_coef = np_polyfit(x_vals, y_vals, 1)
            x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
            fig_scatter.add_trace(go.Scatter(
                x=x_line, y=c + m_coef * x_line,
                name=f"OLS trend (r={corr_total:.2f})",
                line=dict(color="#ef4444", width=1.8, dash="dash")
            ))
        except Exception:
            pass

        fig_scatter.update_layout(
            template="plotly_white", height=430,
            title=f"EU LNG Utilization Rate vs TTF M1 Price  (Pearson r = {corr_total:.2f})",
            xaxis_title="TTF M1 (€/MWh)",
            yaxis_title="Utilization (%)",
            legend=dict(orientation="h", y=-0.14)
        )
        fig_scatter.show()

        # ── Rolling 90-day correlation ─────────────────────────────
        roll_corr = (merged["util"]
                     .rolling(90, min_periods=45)
                     .corr(merged["ttf_m1"]))

        fig_corr = go.Figure()
        fig_corr.add_trace(go.Scatter(
            x=roll_corr.index, y=roll_corr,
            name="90d rolling Pearson r",
            line=dict(color="#6366f1", width=1.8),
            fill="tozeroy", fillcolor="rgba(99,102,241,0.08)"
        ))
        fig_corr.add_hline(y=0, line_color="#94a3b8", line_width=1)
        fig_corr.add_hline(y=0.4,  line_dash="dot", line_color="#22c55e", line_width=1,
                           annotation_text="moderate positive", annotation_position="right")
        fig_corr.add_hline(y=-0.4, line_dash="dot", line_color="#ef4444", line_width=1)

        fig_corr.update_layout(
            template="plotly_white", height=360,
            title="Rolling 90-day Correlation: LNG Utilization vs TTF M1",
            yaxis_title="Pearson r", xaxis_title="",
            yaxis=dict(range=[-1.1, 1.1]),
            legend=dict(orientation="h", y=-0.18)
        )
        fig_corr.show()

        print(f"Overall Pearson r (utilization vs TTF M1): {corr_total:.3f}")
        recent_r = roll_corr.dropna().iloc[-1]
        print(f"90-day rolling r (latest)                : {recent_r:.3f}")
        if abs(corr_total) > 0.3:
            direction = "positive" if corr_total > 0 else "negative"
            print(f"\n→ {direction.capitalize()} correlation is consistent with the economic-pull hypothesis.")
            print(f"  Higher TTF prices attract more LNG cargoes into Europe, raising utilization.")
        else:
            print("\n→ Weak overall correlation — structural long-term contracts may dominate over")
            print("  economic signals in the full sample. Check sub-period results above.")
        print("\nNote: JKM (Asian spot) data not available. TTF M1 is a proxy for the")
        print("European demand signal; a TTF–JKM spread would give a cleaner economic driver.")

## Cell 6 — Send-Out vs Storage Injection: Cross-Correlation

A key policy question is whether LNG send-out is **substituting** for pipeline gas in direct consumption, or **supplementing** storage injection (i.e. LNG is physically filling storage tanks that then serve as buffers). The cross-correlation between daily LNG send-out and gas storage injection at various lags helps clarify the lead-lag relationship.

A positive correlation at **lag 0** suggests same-day co-movement (both rising when demand is high). A positive correlation at **negative lags** (send-out leading injection) would imply LNG regasification feeds the injection stream. A positive correlation at **positive lags** (injection leading send-out) would suggest storage drawdown creating space that LNG then fills.

**Data dependency:** This cell requires `data/processed/eu_aggregate_full.parquet` from notebook 01. If it is unavailable, the cell will skip gracefully.

In [ ]:
# ── Cross-correlation: LNG send-out vs AGSI storage injection ─
storage_path = Path("data/processed/eu_aggregate_full.parquet")

if not storage_path.exists():
    print("⚠️ Gas storage data not found at data/processed/eu_aggregate_full.parquet")
    print("   Run notebook 01_data_ingestion.ipynb first, then re-run this cell.")
elif lng_df.empty or "sendOut" not in lng_df.columns:
    print("⚠️ No LNG send-out data.")
else:
    storage_df = pd.read_parquet(storage_path)
    # Strip timezone
    if hasattr(storage_df.index, "tz") and storage_df.index.tz is not None:
        storage_df.index = storage_df.index.tz_localize(None)
    storage_df.index = pd.to_datetime(storage_df.index)

    # Identify injection column
    inj_col = next((c for c in ["injection", "netFlow", "gasIn"] if c in storage_df.columns), None)
    if inj_col is None:
        print(f"⚠️ No injection column found. Available: {list(storage_df.columns)}")
    else:
        # Merge on date
        merged_xcorr = (
            lng_df[["sendOut"]].rename(columns={"sendOut": "lng_sendout"})
            .join(storage_df[[inj_col]].rename(columns={inj_col: "storage_inj"}), how="inner")
            .dropna()
            .sort_index()
        )

        if merged_xcorr.empty:
            print("⚠️ No overlapping data between LNG and storage.")
        else:
            # Normalise for cross-correlation
            x = (merged_xcorr["lng_sendout"]  - merged_xcorr["lng_sendout"].mean())  / merged_xcorr["lng_sendout"].std()
            y = (merged_xcorr["storage_inj"]  - merged_xcorr["storage_inj"].mean())  / merged_xcorr["storage_inj"].std()

            MAX_LAG = 60
            lags    = range(-MAX_LAG, MAX_LAG + 1)
            xcorr   = [x.corr(y.shift(lag)) for lag in lags]

            # Find peak lag
            peak_idx  = int(np.argmax(np.abs(xcorr)))
            peak_lag  = list(lags)[peak_idx]
            peak_corr = xcorr[peak_idx]

            fig = go.Figure()
            colors_bar = ["#22c55e" if c > 0 else "#ef4444" for c in xcorr]
            fig.add_trace(go.Bar(
                x=list(lags), y=xcorr,
                marker_color=colors_bar,
                name="Cross-correlation"
            ))
            # 95% significance band (approx: ±1.96/sqrt(N))
            sig_band = 1.96 / np.sqrt(len(merged_xcorr))
            fig.add_hline(y=sig_band,  line_dash="dot", line_color="#94a3b8", line_width=1,
                          annotation_text="95% significance", annotation_position="right")
            fig.add_hline(y=-sig_band, line_dash="dot", line_color="#94a3b8", line_width=1)
            fig.add_vline(x=0, line_color="#475569", line_width=1.2)

            fig.update_layout(
                template="plotly_white", height=400,
                title=(f"Cross-Correlation: LNG Send-Out vs Storage Injection  "
                       f"(peak at lag={peak_lag:+d} days, r={peak_corr:.2f})"),
                xaxis_title="Lag (days) — positive = injection leads send-out",
                yaxis_title="Pearson r",
                legend=dict(orientation="h", y=-0.16)
            )
            fig.show()

            # Rolling correlation over time (window=90d at lag 0)
            roll_xcorr = merged_xcorr["lng_sendout"].rolling(90, min_periods=45).corr(
                merged_xcorr["storage_inj"]
            )
            fig2 = go.Figure(go.Scatter(
                x=roll_xcorr.index, y=roll_xcorr,
                name="90d rolling r (lag 0)",
                line=dict(color="#0ea5e9", width=1.8),
                fill="tozeroy", fillcolor="rgba(14,165,233,0.07)"
            ))
            fig2.add_hline(y=0, line_color="#94a3b8", line_width=1)
            fig2.update_layout(
                template="plotly_white", height=340,
                title="Rolling 90-day Correlation: LNG Send-Out vs Storage Injection (lag 0)",
                yaxis_title="Pearson r", xaxis_title="",
                yaxis=dict(range=[-1.1, 1.1]),
                legend=dict(orientation="h", y=-0.2)
            )
            fig2.show()

            print(f"Observations     : {len(merged_xcorr)} overlapping days")
            print(f"Date range       : {merged_xcorr.index.min().date()} → {merged_xcorr.index.max().date()}")
            print(f"Contemporaneous r (lag 0): {xcorr[MAX_LAG]:.3f}")
            print(f"Peak correlation          : r={peak_corr:.3f} at lag={peak_lag:+d} days")
            print(f"95% significance band     : ±{sig_band:.3f}")
            print()
            if abs(peak_lag) <= 3:
                print("→ Peak near lag 0: LNG send-out and storage injection move together,")
                print("  suggesting LNG directly feeds the storage injection stream.")
            elif peak_lag < 0:
                print(f"→ Peak at lag {peak_lag}: LNG send-out leads storage injection by ~{abs(peak_lag)} days,")
                print("  suggesting regasification precedes the recorded injection flow.")
            else:
                print(f"→ Peak at lag {peak_lag}: storage injection leads LNG send-out by ~{peak_lag} days,")
                print("  which may reflect storage drawdown creating pipeline space that LNG then fills.")